# 📖 Lab 1: View Events

Our first functional requirement: **Users should be able to view events.**

When a user navigates to `/event/:eventId` they should see:
- Event name, description, date, and type
- Venue details (name, address, capacity)
- Performer information
- A **seat map** showing which seats are available and which are sold

## 🏗️ Architecture — Starting Point

```
┌────────┐     GET /events/:eventId     ┌───────────────┐         ┌──────────────────┐
│        │ ─────────────────────────────>│               │  SQL    │   PostgreSQL      │
│ Client │                               │ Event Service │────────>│                  │
│        │<─────────────────────────────│               │         │  events           │
└────────┘   Event + Venue + Performer   └───────────────┘         │  venues           │
             + Tickets (for seat map)                              │  performers       │
                                                                   │  tickets          │
                                                                   └──────────────────┘
```

This is the simplest possible setup. No API Gateway, no caching, no load balancing — just a service reading from a database.

## Learning Objectives

- Understand how the view event flow works end-to-end
- See how entities are stored and joined in PostgreSQL
- Build the query that powers the `GET /events/:eventId` API
- Render a simple seat map from ticket data

## 🛠️ Setup

### 1. Start PostgreSQL

```bash
cd system-designs/ticketmaster
docker-compose up -d
```

### 2. Create a virtual environment & install dependencies

```bash
python -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```

### 3. Register & select the Jupyter kernel

Register the venv as a kernel so VS Code can find it:

```bash
python -m ipykernel install --user --name ticketmaster --display-name "Ticketmaster (Python)"
```

Then in VS Code, click the **kernel picker** (top-right of the notebook) and select **"Ticketmaster (Python)"**. If you don't see it, reload the window (`Cmd+Shift+P` → "Reload Window").

### 4. Browse the data (optional)

Open **Adminer** at [http://localhost:8080](http://localhost:8080):
- System: `PostgreSQL` | Server: `postgres` | User: `demo` | Password: `demo` | Database: `ticketmaster`

In [5]:
import psycopg2
import psycopg2.extras
import json

DB_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "user": "demo",
    "password": "demo",
    "database": "ticketmaster",
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

# Quick connection test
conn = get_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM events")
print(f"✅ Connected! Found {cur.fetchone()[0]} events in the database.")
cur.close()
conn.close()

✅ Connected! Found 5 events in the database.


## 📊 Exploring the Data Model

Before writing the view endpoint logic, let's see what's in the database. Our `init.sql` created these tables:

| Table | Purpose |
|-------|---------|
| `performers` | Artists, bands, teams |
| `venues` | Locations with a JSON `seat_map` |
| `events` | A performer at a venue on a date |
| `tickets` | One row per seat per event (`available` or `sold`) |
| `bookings` | Groups tickets into a purchase (used later) |

Let's peek at each one.

In [8]:
# Let's see what events we have
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("SELECT id, name, event_type, event_date, venue_id, performer_id FROM events ORDER BY event_date")
events = cur.fetchall()

print("🎫 Events in the database:\n")
print(f"{'ID':<4} {'Name':<35} {'Type':<10} {'Date':<20} {'Venue ID':<10} {'Performer ID'}")
print("-" * 100)
for e in events:
    print(f"{e['id']:<4} {e['name']:<35} {e['event_type']:<10} {str(e['event_date']):<20} {e['venue_id']:<10} {e['performer_id']}")

cur.close()
conn.close()

🎫 Events in the database:

ID   Name                                Type       Date                 Venue ID   Performer ID
----------------------------------------------------------------------------------------------------
1    The Eras Tour - NYC                 concert    2026-06-15 20:00:00  1          1
2    Championship Game                   sports     2026-07-01 19:30:00  2          5
3    Renaissance World Tour              concert    2026-08-10 21:00:00  3          4
4    Kendrick Lamar: Big Steppers        concert    2026-09-20 20:00:00  1          2
5    Music of the Spheres                concert    2026-10-05 19:00:00  2          3


In [9]:
# Venues and their seat maps
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("SELECT id, name, city, country, capacity FROM venues")
venues = cur.fetchall()

print("🏟️ Venues:\n")
print(f"{'ID':<4} {'Name':<30} {'City':<15} {'Country':<10} {'Capacity'}")
print("-" * 75)
for v in venues:
    print(f"{v['id']:<4} {v['name']:<30} {v['city']:<15} {v['country']:<10} {v['capacity']}")

cur.close()
conn.close()

🏟️ Venues:

ID   Name                           City            Country    Capacity
---------------------------------------------------------------------------
1    Madison Square Garden          New York        US         20000
2    SoFi Stadium                   Inglewood       US         70000
3    The O2 Arena                   London          UK         20000


In [10]:
# Let's look at the seat map structure for MSG
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("SELECT seat_map FROM venues WHERE id = 1")
seat_map = cur.fetchone()["seat_map"]

print("🗺️ Madison Square Garden — Seat Map Structure:\n")
print(json.dumps(seat_map, indent=2))

cur.close()
conn.close()

🗺️ Madison Square Garden — Seat Map Structure:

{
  "sections": [
    {
      "name": "FLOOR",
      "rows": [
        {
          "label": "A",
          "seats": 10
        },
        {
          "label": "B",
          "seats": 10
        },
        {
          "label": "C",
          "seats": 10
        }
      ]
    },
    {
      "name": "LOWER",
      "rows": [
        {
          "label": "A",
          "seats": 15
        },
        {
          "label": "B",
          "seats": 15
        },
        {
          "label": "C",
          "seats": 15
        }
      ]
    },
    {
      "name": "UPPER",
      "rows": [
        {
          "label": "A",
          "seats": 20
        },
        {
          "label": "B",
          "seats": 20
        }
      ]
    }
  ]
}


In [11]:
# Ticket counts per event
conn = get_connection()
cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

cur.execute("""
    SELECT 
        e.name AS event_name,
        COUNT(*) AS total_tickets,
        COUNT(*) FILTER (WHERE t.status = 'available') AS available,
        COUNT(*) FILTER (WHERE t.status = 'sold') AS sold
    FROM tickets t
    JOIN events e ON e.id = t.event_id
    GROUP BY e.name
    ORDER BY e.name
""")
rows = cur.fetchall()

print("🎟️ Ticket breakdown per event:\n")
print(f"{'Event':<35} {'Total':<8} {'Available':<12} {'Sold'}")
print("-" * 65)
for r in rows:
    print(f"{r['event_name']:<35} {r['total_tickets']:<8} {r['available']:<12} {r['sold']}")

cur.close()
conn.close()

🎟️ Ticket breakdown per event:

Event                               Total    Available    Sold
-----------------------------------------------------------------
Championship Game                   134      104          30
Kendrick Lamar: Big Steppers        115      115          0
Music of the Spheres                134      134          0
Renaissance World Tour              70       55           15
The Eras Tour - NYC                 115      90           25


## 🔧 Building the Event Service: `GET /events/:eventId`

Now let's build the actual logic that powers the view event API.

When a user hits `GET /events/:eventId`, the Event Service needs to:
1. Fetch the **event** by ID
2. Join in the **venue** and **performer** data
3. Fetch all **tickets** for that event (so the client can render the seat map)
4. Return it all as a single response

This is a classic read operation — one query that joins across multiple tables.

In [12]:
def get_event(event_id: int) -> dict:
    """
    Simulates the Event Service handler for GET /events/:eventId.
    
    Fetches event + venue + performer in one query,
    then fetches all tickets for the seat map.
    """
    conn = get_connection()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)

    # Step 1: Fetch event with venue and performer (single JOIN query)
    cur.execute("""
        SELECT 
            e.id AS event_id,
            e.name AS event_name,
            e.description AS event_description,
            e.event_type,
            e.event_date,
            e.status AS event_status,
            v.id AS venue_id,
            v.name AS venue_name,
            v.address AS venue_address,
            v.city AS venue_city,
            v.state AS venue_state,
            v.country AS venue_country,
            v.capacity AS venue_capacity,
            v.seat_map,
            p.id AS performer_id,
            p.name AS performer_name,
            p.description AS performer_description,
            p.genre AS performer_genre,
            p.image_url AS performer_image_url
        FROM events e
        JOIN venues v ON e.venue_id = v.id
        JOIN performers p ON e.performer_id = p.id
        WHERE e.id = %s
    """, (event_id,))

    row = cur.fetchone()
    if not row:
        cur.close()
        conn.close()
        return None

    # Step 2: Fetch all tickets for the event (for seat map rendering)
    cur.execute("""
        SELECT id, section, row_label, seat_number, price, status
        FROM tickets
        WHERE event_id = %s
        ORDER BY section, row_label, seat_number
    """, (event_id,))

    tickets = cur.fetchall()

    cur.close()
    conn.close()

    # Step 3: Shape the response (what the API returns to the client)
    return {
        "event": {
            "id": row["event_id"],
            "name": row["event_name"],
            "description": row["event_description"],
            "type": row["event_type"],
            "date": str(row["event_date"]),
            "status": row["event_status"],
        },
        "venue": {
            "id": row["venue_id"],
            "name": row["venue_name"],
            "address": row["venue_address"],
            "city": row["venue_city"],
            "state": row["venue_state"],
            "country": row["venue_country"],
            "capacity": row["venue_capacity"],
            "seatMap": row["seat_map"],
        },
        "performer": {
            "id": row["performer_id"],
            "name": row["performer_name"],
            "description": row["performer_description"],
            "genre": row["performer_genre"],
            "imageUrl": row["performer_image_url"],
        },
        "tickets": [dict(t) for t in tickets],
    }

print("✅ get_event() function defined.")

✅ get_event() function defined.


In [13]:
# Let's call it! View "The Eras Tour - NYC" (event_id = 1)
response = get_event(event_id=1)

print("=" * 60)
print(f"🎵 {response['event']['name']}")
print("=" * 60)
print(f"\n📝 {response['event']['description']}")
print(f"📅 {response['event']['date']}")
print(f"🎤 {response['performer']['name']} ({response['performer']['genre']})")
print(f"🏟️  {response['venue']['name']}, {response['venue']['city']}, {response['venue']['state']}")
print(f"💺 Venue capacity: {response['venue']['capacity']:,}")

total = len(response["tickets"])
available = sum(1 for t in response["tickets"] if t["status"] == "available")
sold = total - available
print(f"\n🎟️  Tickets: {total} total | {available} available | {sold} sold")

🎵 The Eras Tour - NYC

📝 Taylor Swift live at MSG. A journey through every musical era.
📅 2026-06-15 20:00:00
🎤 Taylor Swift (Pop)
🏟️  Madison Square Garden, New York, NY
💺 Venue capacity: 20,000

🎟️  Tickets: 115 total | 90 available | 25 sold


## 🗺️ Rendering the Seat Map

The client needs to render an interactive seat map. It combines two pieces of data:

1. **Venue seat map** (from `venues.seat_map`) — defines the layout: sections, rows, and how many seats per row
2. **Ticket statuses** (from `tickets`) — tells us which seats are `available` vs `sold`

Below we render a simple text-based seat map to visualize what the client would display.

- `○` = available seat
- `●` = sold seat

In [14]:
def render_seat_map(response: dict):
    """
    Renders a text-based seat map from the API response.
    In production, the client would use the seat_map coordinates
    to render an interactive SVG/Canvas UI.
    """
    tickets = response["tickets"]

    # Build a lookup: (section, row, seat) -> status
    status_map = {}
    for t in tickets:
        key = (t["section"], t["row_label"], t["seat_number"])
        status_map[key] = t["status"]

    seat_map = response["venue"]["seatMap"]

    print(f"🗺️  Seat Map: {response['venue']['name']}")
    print(f"   Event: {response['event']['name']}\n")

    for section in seat_map["sections"]:
        section_name = section["name"]
        print(f"  ┌{'─' * 40}┐")
        print(f"  │  Section: {section_name:<28}│")
        print(f"  ├{'─' * 40}┤")
        for row in section["rows"]:
            row_label = row["label"]
            num_seats = row["seats"]
            seats_display = ""
            for seat_num in range(1, num_seats + 1):
                status = status_map.get((section_name, row_label, seat_num), "available")
                seats_display += "● " if status == "sold" else "○ "
            print(f"  │  Row {row_label}: {seats_display.strip():<30}│")
        print(f"  └{'─' * 40}┘\n")

    print("  Legend: ○ = available  ● = sold")

render_seat_map(response)

🗺️  Seat Map: Madison Square Garden
   Event: The Eras Tour - NYC

  ┌────────────────────────────────────────┐
  │  Section: FLOOR                       │
  ├────────────────────────────────────────┤
  │  Row A: ○ ○ ○ ○ ○ ● ○ ○ ● ○           │
  │  Row B: ○ ○ ○ ○ ○ ○ ○ ○ ○ ○           │
  │  Row C: ○ ○ ● ○ ○ ○ ○ ○ ○ ●           │
  └────────────────────────────────────────┘

  ┌────────────────────────────────────────┐
  │  Section: LOWER                       │
  ├────────────────────────────────────────┤
  │  Row A: ○ ○ ○ ○ ● ○ ○ ● ● ○ ○ ● ○ ○ ○ │
  │  Row B: ○ ○ ● ○ ○ ○ ● ○ ○ ○ ○ ○ ○ ○ ○ │
  │  Row C: ○ ● ○ ○ ○ ● ○ ● ● ○ ○ ○ ○ ● ● │
  └────────────────────────────────────────┘

  ┌────────────────────────────────────────┐
  │  Section: UPPER                       │
  ├────────────────────────────────────────┤
  │  Row A: ○ ● ○ ○ ○ ○ ○ ○ ○ ○ ○ ● ○ ● ● ○ ○ ○ ○ ○│
  │  Row B: ○ ○ ○ ○ ● ○ ○ ● ● ○ ● ○ ○ ● ○ ○ ○ ○ ○ ○│
  └────────────────────────────────────────┘

  Legend: ○ = available

## 🏗️ Architecture — After Lab 1

We now have the first piece of our system:

```
┌────────┐         ┌─────────────┐         ┌───────────────┐         ┌──────────────────┐
│        │  HTTP   │ API Gateway │  route   │               │  SQL    │   PostgreSQL      │
│ Client │────────>│ - auth      │────────> │ Event Service │────────>│                  │
│        │<────────│ - rate limit│<────────│               │<────────│  events           │
└────────┘         │ - routing   │         └───────────────┘         │  venues           │
                   └─────────────┘                                    │  performers       │
                                                                      │  tickets          │
                                                                      └──────────────────┘
```

**What we built:** Event Service reads event + venue + performer + tickets in 2 SQL queries (one JOIN, one ticket fetch) and returns a combined response.

**What's missing:** Search, booking, caching, scaling — all coming in the next labs.

## 🔍 Under the Hood: Query Performance

Let's see how PostgreSQL executes our queries. This matters because in the real system this endpoint gets hammered — every user viewing an event page triggers this query.

We use `EXPLAIN ANALYZE` to see the **actual** execution plan.